# 00 — Extracción y exploración inicial

Este notebook extrae los archivos comprimidos de los chats de Conti e inspecciona su formato real antes de construir los parsers.

## 0. Prerequisitos

Ejecutar en terminal si no está instalado:
```bash
sudo apt install p7zip-full
```

In [1]:
import subprocess, shutil, os

if not shutil.which('7z'):
    raise EnvironmentError('p7zip-full no está instalado. Ejecuta: sudo apt install p7zip-full')

print('7z disponible:', shutil.which('7z'))

7z disponible: /usr/bin/7z


## 1. Rutas

In [2]:
from pathlib import Path

RANSOMWARE_DIR = Path('/home/drjekyll/Documentos/umbrella/data_bruto/Ransomware')
CONTI_ZIP      = RANSOMWARE_DIR / 'Conti_Chats_2022.zip'
RAW_DIR        = Path('../data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

print('ZIP origen:', CONTI_ZIP)
print('Directorio destino:', RAW_DIR.resolve())
print('ZIP existe:', CONTI_ZIP.exists())

ZIP origen: /home/drjekyll/Documentos/umbrella/data_bruto/Ransomware/Conti_Chats_2022.zip
Directorio destino: /home/drjekyll/Documentos/umbrella/FearOfTheDark/ContiLeaks/data/raw
ZIP existe: True


## 2. Extraer Conti_Chats_2022.zip (contiene 3 archivos .7z)

In [3]:
# Extraer el ZIP exterior
result = subprocess.run(
    ['7z', 'x', str(CONTI_ZIP), f'-o{RAW_DIR}', '-y'],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)


7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=es_ES.UTF-8 Threads:32 OPEN_MAX:4096

Scanning the drive for archives:
1 file, 6948300 bytes (6786 KiB)

Extracting archive: /home/drjekyll/Documentos/umbrella/data_bruto/Ransomware/Conti_Chats_2022.zip
--
Path = /home/drjekyll/Documentos/umbrella/data_bruto/Ransomware/Conti_Chats_2022.zip
Type = zip
Physical Size = 6948300

Everything is Ok

Files: 3
Size:       6947447
Compressed: 6948300



In [4]:
# Ver qué hay en raw/ ahora
for p in sorted(RAW_DIR.rglob('*')):
    size = p.stat().st_size if p.is_file() else 0
    print(f'{"DIR" if p.is_dir() else f"{size/1024:.0f} KB":>10}  {p.relative_to(RAW_DIR)}')

   2361 KB  Conti Chat Logs 2020.7z
   1132 KB  Conti Jabber Chat Logs 2021 - 2022.7z
   3292 KB  Conti Rocket Chat Leaks.7z


## 3. Extraer los 3 archivos .7z internos

In [5]:
sevenz_files = list(RAW_DIR.rglob('*.7z'))
print(f'Archivos .7z encontrados: {len(sevenz_files)}')
for f in sevenz_files:
    print(' -', f.name)

Archivos .7z encontrados: 3
 - Conti Chat Logs 2020.7z
 - Conti Jabber Chat Logs 2021 - 2022.7z
 - Conti Rocket Chat Leaks.7z


In [6]:
# Extraer cada .7z en su propio subdirectorio
for archive in sevenz_files:
    dest = RAW_DIR / archive.stem
    dest.mkdir(exist_ok=True)
    result = subprocess.run(
        ['7z', 'x', str(archive), f'-o{dest}', '-y'],
        capture_output=True, text=True
    )
    status = 'OK' if result.returncode == 0 else 'ERROR'
    print(f'[{status}] {archive.name} → {dest.name}/')
    if result.returncode != 0:
        print('  STDERR:', result.stderr[:300])

[OK] Conti Chat Logs 2020.7z → Conti Chat Logs 2020/
[OK] Conti Jabber Chat Logs 2021 - 2022.7z → Conti Jabber Chat Logs 2021 - 2022/
[OK] Conti Rocket Chat Leaks.7z → Conti Rocket Chat Leaks/


## 4. Inventario completo de archivos extraídos

In [7]:
import pandas as pd

inventory = []
for p in sorted(RAW_DIR.rglob('*')):
    if p.is_file() and p.suffix.lower() not in ('.7z', '.zip'):
        inventory.append({
            'archivo': p.relative_to(RAW_DIR),
            'extension': p.suffix.lower(),
            'tamaño_MB': round(p.stat().st_size / 1024**2, 2)
        })

df_inv = pd.DataFrame(inventory)
display(df_inv)

,archivo,extension,tamaño_MB
0,Conti Chat Logs 2020/Conti Chat Logs 2020/185....,.json,0.39
1,Conti Chat Logs 2020/Conti Chat Logs 2020/185....,.json,0.15
2,Conti Chat Logs 2020/Conti Chat Logs 2020/185....,.json,0.24
3,Conti Chat Logs 2020/Conti Chat Logs 2020/185....,.json,0.30
4,Conti Chat Logs 2020/Conti Chat Logs 2020/185....,.json,0.19
...,...,...,...
2097,Conti Rocket Chat Leaks/Conti Rocket Chat Leak...,.json,0.00
2098,Conti Rocket Chat Leaks/Conti Rocket Chat Leak...,.json,0.00
2099,Conti Rocket Chat Leaks/Conti Rocket Chat Leak...,.json,0.01
2100,Conti Rocket Chat Leaks/Conti Rocket Chat Leak...,.json,0.00


## 5. Inspeccionar las primeras líneas de cada archivo

In [8]:
for _, row in df_inv.iterrows():
    fpath = RAW_DIR / row['archivo']
    print(f"\n{'='*60}")
    print(f"ARCHIVO: {row['archivo']}  ({row['tamaño_MB']} MB)")
    print('='*60)
    try:
        with open(fpath, 'r', encoding='utf-8', errors='replace') as f:
            for i, line in enumerate(f):
                if i >= 15:
                    break
                print(repr(line[:200]))
    except Exception as e:
        print(f'No se pudo leer como texto: {e}')


ARCHIVO: Conti Chat Logs 2020/Conti Chat Logs 2020/185.25.51.173-20200622.json  (0.39 MB)
'{\n'
'  "ts": "2020-06-21T15:09:47.158053",\n'
'  "from": "price@q3mcco35auwcstmt.onion",\n'
'  "to": "mentos@q3mcco35auwcstmt.onion",\n'
'  "body": "привет"\n'
'}\n'
'{\n'
'  "ts": "2020-06-21T15:21:09.214052",\n'
'  "from": "mentos@q3mcco35auwcstmt.onion",\n'
'  "to": "price@q3mcco35auwcstmt.onion",\n'
'  "body": "привет"\n'
'}\n'
'{\n'
'  "ts": "2020-06-21T15:21:33.055319",\n'
'  "from": "price@q3mcco35auwcstmt.onion",\n'

ARCHIVO: Conti Chat Logs 2020/Conti Chat Logs 2020/185.25.51.173-20200623.json  (0.15 MB)
'{\n'
'  "ts": "2020-06-23T04:14:43.377922",\n'
'  "from": "bill@q3mcco35auwcstmt.onion",\n'
'  "to": "buza@q3mcco35auwcstmt.onion",\n'
'  "body": "привет"\n'
'}\n'
'{\n'
'  "ts": "2020-06-23T04:14:44.632948",\n'
'  "from": "bill@q3mcco35auwcstmt.onion",\n'
'  "to": "buza@q3mcco35auwcstmt.onion",\n'
'  "body": "https://mk6gwg6mwnn6if33.onion/issues/108/edit"\n'
'}\n'
'{\n'
'  "ts": "20

## 6. EDA básica por archivo

In [9]:
# Contar líneas de cada archivo (indicativo de volumen)
for _, row in df_inv.iterrows():
    fpath = RAW_DIR / row['archivo']
    try:
        with open(fpath, 'r', encoding='utf-8', errors='replace') as f:
            n_lines = sum(1 for _ in f)
        print(f"{n_lines:>8} líneas  {row['archivo']}")
    except Exception as e:
        print(f'         ERROR   {row["archivo"]}: {e}')

   10386 líneas  Conti Chat Logs 2020/Conti Chat Logs 2020/185.25.51.173-20200622.json
    4290 líneas  Conti Chat Logs 2020/Conti Chat Logs 2020/185.25.51.173-20200623.json
    5490 líneas  Conti Chat Logs 2020/Conti Chat Logs 2020/185.25.51.173-20200624.json
    9366 líneas  Conti Chat Logs 2020/Conti Chat Logs 2020/185.25.51.173-20200625.json
    5790 líneas  Conti Chat Logs 2020/Conti Chat Logs 2020/185.25.51.173-20200626.json
     138 líneas  Conti Chat Logs 2020/Conti Chat Logs 2020/185.25.51.173-20200627.json
     792 líneas  Conti Chat Logs 2020/Conti Chat Logs 2020/185.25.51.173-20200628.json
    5814 líneas  Conti Chat Logs 2020/Conti Chat Logs 2020/185.25.51.173-20200629.json
    6414 líneas  Conti Chat Logs 2020/Conti Chat Logs 2020/185.25.51.173-20200630.json
    3378 líneas  Conti Chat Logs 2020/Conti Chat Logs 2020/185.25.51.173-20200701.json
    2670 líneas  Conti Chat Logs 2020/Conti Chat Logs 2020/185.25.51.173-20200702.json
    2784 líneas  Conti Chat Logs 2020/Conti

## 7. Conclusiones

Completar manualmente tras ejecutar las celdas anteriores:

| Fuente | Archivo | Formato detectado | N líneas | Parser a usar |
|---|---|---|---|---|
| Jabber 2020 | ... | TXT / XML / JSON | ... | parse_txt / parse_jabber_xml / parse_rocketchat_json |
| Jabber 2021-2022 | ... | TXT / XML / JSON | ... | ... |
| Rocket.Chat | ... | TXT / XML / JSON | ... | ... |